In [ ]:
# Pipeline to slice catalogue into volume limited redshift slices and compare LF estimates


import catalogue_analysis as ca
import numpy as np
from astropy.table import join,Table,Column,vstack
from astropy.io import fits
#
# Specify regions to loop over
regions=('S','N')
#
# Read the enhanced BGS catalogue produced by Augment_BGS_cat.ipynb
dat=Table.read('BGS_Y3_Sept2025.fits')
print('Meta data stored with ths table:')
print(dat.meta)
#
# Check selection cuts agree with stored meta data of the sample we have read
Sel=ca.selection('S')
assert dat.meta['FAINT_S'] == Sel['faint'], f"Faint limits do not match: {dat.meta['FAINT_S']} != {Sel['faint']}"
assert dat.meta['BRIGHT_S'] == Sel['bright'], f"Bright limits do not match: {dat.meta['BRIGHT_S']} != {Sel['bright']}"
assert dat.meta['ZMAX_S'] == Sel['zmax'], f"Zmax do not match: {dat.meta['ZMAX_S']} != {Sel['zmax']}"
assert dat.meta['ZMIN_S'] == Sel['zmin'], f"Zmin do not match: {dat.meta['ZMIN_S']} != {Sel['zmin']}"
assert dat.meta['QEVOL_S'] == Sel['Qevol'], f"Qevol do not match: {dat.meta['QEVOL_S']} != {Sel['Qevol']}"
assert dat.meta['AREA_S'] == Sel['area'], f"Areas do not match: {dat.meta['AREA_S']} != {Sel['area']}"
assert dat.meta['F_RAN_S'] == Sel['f_ran'], f"f_ran do not match: {dat.meta['F_RAN_S']} != {Sel['f_ran']}"
assert dat.meta['COL_S'] == Sel['col'], f"Colours do not match: {dat.meta['COL_S']} != {Sel['col']}"
assert dat.meta['STYLE_S'] == Sel['style'], f"Styles do not match: {dat.meta['STYLE_S']} != {Sel['style']}"
Sel=ca.selection('N')
assert dat.meta['FAINT_N'] == Sel['faint'], f"Faint limits do not match: {dat.meta['FAINT_N']} != {Sel['faint']}"
assert dat.meta['BRIGHT_N'] == Sel['bright'], f"Bright limits do not match: {dat.meta['BRIGHT_N']} != {Sel['bright']}"
assert dat.meta['ZMAX_N'] == Sel['zmax'], f"Zmax do not match: {dat.meta['ZMAX_N']} != {Sel['zmax']}"
assert dat.meta['ZMIN_N'] == Sel['zmin'], f"Zmin do not match: {dat.meta['ZMIN_N']} != {Sel['zmin']}"
assert dat.meta['QEVOL_N'] == Sel['Qevol'], f"Qevol do not match: {dat.meta['QEVOL_N']} != {Sel['Qevol']}"
assert dat.meta['AREA_N'] == Sel['area'], f"Areas do not match: {dat.meta['AREA_N']} != {Sel['area']}"
assert dat.meta['F_RAN_N'] == Sel['f_ran'], f"f_ran do not match: {dat.meta['F_RAN_N']} != {Sel['f_ran']}"
assert dat.meta['COL_N'] == Sel['col'], f"Colours do not match: {dat.meta['COL_N']} != {Sel['col']}"
assert dat.meta['STYLE_N'] == Sel['style'], f"Styles do not match: {dat.meta['STYLE_N']} != {Sel['style']}"

# Check cosmology used matches with that used to create the catalogue
assert dat.meta['OM0'] == ca.cosmo.Om0, f"Omega_0 does not match: {dat.meta['OM0']} != {ca.cosmo.Om0}"
assert dat.meta['H0'] == ca.cosmo.H0.value, f"H_0 does not match: {dat.meta['H0']} != {ca.cosmo.H0.value}"
assert dat.meta['TCMB0'] == ca.cosmo.Tcmb0.value, f"Omega_0 does not match: {dat.meta['OM0']} != {ca.cosmo.Tcmb0.value}"

print('Statistics of the data in the table:')
#Placeholders for additional columns to add to the tbale 
dat.add_column(Column(name='izbin', data=np.zeros(dat['Z'].size))) # redshift label bin
dat.add_column(Column(name='invollim', format='L',data=np.zeros(dat['Z'].size,dtype='bool'))) # flag if in the volume limited subset of this redshift bin
dat.add_column(Column(name='vollim', data=np.zeros(dat['Z'].size))) # volume of redshift slice it is in
dat.add_column(Column(name='compl', format='L', data=np.zeros(dat['Z'].size),dtype='bool')) # flag if within colour complete sample
dat.info('stats')

 

In [ ]:
# Read one the corresponding Y3 random (It is matched to the orginal clustering catalogue and not yet the cuts made to our sample
fpath='/global/cfs/cdirs/desi/survey/catalogs/DA2/LSS/loa-v1/LSScats/test/nonKP/BGS_BRIGHT_7_clustering.ran.fits'
ranfull=ca.Y3load_catalogues(fpath)


print('Full Random Catalogue statistics:')
ranfull.info('stats')


#Random sample and apply all cuts from the start for a quick run through
Sel=ca.selection('N')# import North selection cuts  
smaskN= (ranfull['Z'] > Sel['zmin']) & (ranfull['Z'] < Sel['zmax']) & (np.random.rand(ranfull['Z'].size)<Sel['f_ran'])\
      & (ranfull['rmag'] < Sel['faint'])  & (ranfull['rmag'] > Sel['bright']) & (ranfull['reg']=='N') 
Sel=ca.selection('S') # import South selection cuts    
smaskS= (ranfull['Z'] > Sel['zmin']) & (ranfull['Z'] < Sel['zmax']) &(np.random.rand(ranfull['Z'].size)<Sel['f_ran']) \
      & (ranfull['rmag'] < Sel['faint'])  & (ranfull['rmag'] > Sel['bright']) & (ranfull['reg']=='S')
smask= np.logical_or(smaskN,smaskS) #True for objects making cuts in North or South
ran=ranfull[smask]    
print('Statistics after applying sample selection cuts:')
print('South sample size:',smaskS.sum())
print('North sample size:',smaskN.sum())
ran.info('stats')


In [ ]:
# Set up jackknife region limits to equally split up the randoms provided
# and assign unique indices to each region for both data and randoms
njack_S=0
dat['ijack']=-999  # default to mean unassigned
ran['ijack']=-999  # default to mean unassigned
for reg in regions:
    regmask = (dat['reg']==reg)   
    ranregmask = (ran['reg']==reg)  
    print('Processing region ',reg)
    if (reg=='S'):
      # set up the RA and dec boundaries  
      njack_S,limits_S=ca.solve_jackknife_nonsq(ran[ranregmask], ndiv_ra=16, ndiv_dec=5, offset=275)
      # assign randoms to regions
      ca.set_jackknife(ran, ranregmask, limits_S, 0, njack_S)
      # assign data to the same regions
      ca.set_jackknife(dat, regmask, limits_S, 0, njack_S)
    else: 
      # set up the RA and dec boundaries   
      njack_N,limits_N=ca.solve_jackknife_nonsq(ran[ranregmask], ndiv_ra=12, ndiv_dec=3, offset=275)
      # assign randoms to regions
      ca.set_jackknife(ran, ranregmask, limits_N, njack_S, njack_N)
      # assign data to the same regions  
      ca.set_jackknife(dat, regmask, limits_N, njack_S, njack_N)
    

In [ ]:
# Visualise the jackknife regions 'Black crosses indicate any unassigned objects'
ca.sky_plot_jack(dat)
ca.sky_plot_jack(ran)

In [ ]:
# Define redshift slices and volume limited samples within them

# set up the boundaries of the redshift slices with the lowest limit set by the already imposed Sel['zmin']
zbin_edges= np.array([Sel['zmin'],0.05,0.1,0.2,0.3,0.4,0.5,0.6])
print('redshift cuts:',zbin_edges)


# Label according to redshift sample and compute corresponding volumes
# Also returns the bounding absolute magnitudes within which each sample is complete
faint_bound,bright_bound=ca.redshiftslices(dat,zbin_edges,regions,plotfrac=1.0)


In [ ]:
import matplotlib.pyplot as plt
# premlinary call to set up magnitude bins
print('Preliminary call to set up magnitude bins and set up consistentlty sized arrays to store the results.')
_,_,_,magbins=ca.lumfun_vmax(dat,regions,ratio=False,Vollim=False,plot=False)

# create placeholders for the lF estimates from each redshift slice and each region
log_phi_vlim=np.zeros((zbin_edges.size-1, 2, magbins.size))
log_phi_vlim_hi=np.zeros((zbin_edges.size-1, 2, magbins.size))
log_phi_vlim_low=np.zeros((zbin_edges.size-1, 2, magbins.size))
log_phi_vmax=np.zeros((2, magbins.size))
log_phi_vmax_hi=np.zeros((2, magbins.size))
log_phi_vmax_low=np.zeros((2, magbins.size))

#Loop over regions
ireg = -1
for reg in regions:
    ireg += 1
    print('Making LF estimates for all redshift slices in region ', ireg,'=',reg) 
    #overall standard 1/Vmax LFs
    log_phi_vmax[ireg,:],log_phi_vmax_low[ireg,:],log_phi_vmax_hi[ireg,:],magbins=ca.lumfun_vmax(dat,reg,ratio=False,Vollim=False,plot=False)
 
    # Now estimate LF's from each redshift sample
    #loop over the redshift bins
    for jzbin in range(0,zbin_edges.size-1):  
      mask=(dat['izbin']==jzbin) & dat['invollim'] #select the volume limited subset of this redshift bin
      print('zbin=',jzbin,'sample size:',mask.sum())
      log_phi_vlim[jzbin,ireg,:],log_phi_vlim_low[jzbin,ireg,:],log_phi_vlim_hi[jzbin,ireg,:],magbins=ca.lumfun_vmax(dat[mask],reg,ratio=False,Vollim=True,plot=False)  




print('All LFs computed')

In [ ]:
# Replot all the estimates
print('Replotting')
fig = plt.figure()         #defining so legend can be shifted  
ax = fig.add_subplot(111)

# Reference Schechter function from Sam's draft paper Table 1
phi_star=10.0**(-2.13)
alpha=-1.36
mstar=-21.10
l_lstar= 10.0**(0.4*(-magbins+mstar))
phi_sch= phi_star * l_lstar**(1.0+alpha) * np.exp(-l_lstar) *(np.log(10.0)/2.5)
log_phi_sch = np.log10(phi_sch)
plt.plot(magbins,log_phi_sch, label='Reference Schechter Function')

ireg = -1
col=('magenta','cyan') # colours for S and N overall 1/Vmax LFs
colz=('red','blue') # colours for S and N redshift slices
for reg in regions:
    ireg += 1
    #Replot the overall 1/Vmax LF estimate
    plt.fill_between(magbins,log_phi_vmax_low[ireg,:],log_phi_vmax_hi[ireg,:],color=col[ireg],alpha=0.5,label="$1/V_{max}$")

    #loop over redshift bins
    for jzbin in range(0,zbin_edges.size-1):  
        zlabel=f"Vol lim:{zbin_edges[jzbin]}<z<:{zbin_edges[jzbin+1]}"
        plt.fill_between(magbins,log_phi_vlim_low[jzbin,ireg,:],log_phi_vlim_hi[jzbin,ireg,:],color=colz[ireg],alpha=0.5,label=zlabel)

plt.xlabel('$M_r - 5 log h$')
plt.ylabel('$log_{10} \phi(M_r)\quad  [mag^{-1} (Mpc/h)^{-3}]$')
plt.xlim([-24,-13])
plt.ylim([-7,0.0])
leg=plt.legend()

plt.draw() # Draw the figure so you can find the positon of the legend. 
# Get the bounding box of the original legend
bb = leg.get_bbox_to_anchor().transformed(ax.transAxes.inverted()) 
# Change to location of the legend. 
xOffset = 0.6
bb.x0 += xOffset
bb.x1 += xOffset
leg.set_bbox_to_anchor(bb, transform = ax.transAxes)
plt.show()

# Replot all the estimates without errors
print('Replotting without errors and boosts and shifts')
fig = plt.figure()         #defining so legend can be shifted  
ax = fig.add_subplot(111)

# Reference Schechter function from Sam's draft paper Table 1
plt.plot(magbins,log_phi_sch, label='Reference Schechter Function')


dmag=magbins[1]-magbins[0] # magnitude bin spacing
ireg = -1
style=('solid','dotted','dashed','dashdot','solid','dotted','dashed')
for reg in regions:
    ireg += 1
    # Replot the overall 1/Vmax LF estimate
    plt.plot(magbins,log_phi_vmax[ireg,:],color=col[ireg],alpha=0.5,label="$1/V_{max}$")
  
    

    # loop over redshift bins
    for jzbin in range(0,zbin_edges.size-1):  
        zlabel=f"Vol lim:{zbin_edges[jzbin]}<z<:{zbin_edges[jzbin+1]}"

        
        # Determine completeness of each magnitude bin from the bounding absolute magnitudes of this redshift slice

        #print('region',reg)
        #print('jzbin',jzbin)
        #print('bounds:',bright_bound[ireg,jzbin],faint_bound[ireg,jzbin])
        maghi=np.clip(magbins+0.5*dmag,-100.0,faint_bound[ireg,jzbin]) # hi-edge of occupied portion of the bin
        maglo=np.clip(magbins-0.5*dmag,bright_bound[ireg,jzbin],100.0) # lo-edge of occupied portion of the bin
        compl=np.clip((maghi-maglo)/dmag,0.0,1.0) #fraction of the bin occupied
      
        magbins_shifted=np.copy(magbins)
        notcomplete=(compl<1.0) & (compl>0.0)
        magbins_shifted[notcomplete]=0.5*(maghi[notcomplete]+maglo[notcomplete]) # just shift the partially complete bins
       
        magshifts=(magbins-magbins_shifted)/dmag
        # mask empty bins and don't plot them  
        mask=compl>0.0
        #print('magbinshifts',magshifts[mask])   
        #print('boosts:',-np.log10(compl[mask]))
        plt.plot(magbins_shifted[mask],log_phi_vlim[jzbin,ireg,:][mask]-np.log10(compl[mask]),color=colz[ireg],alpha=0.5,label=zlabel,linestyle=style[jzbin])
       

plt.xlabel('$M_r - 5 log h$')
plt.ylabel('$log_{10} \phi(M_r)\quad  [mag^{-1} (Mpc/h)^{-3}]$')
plt.xlim([-24,-13])
plt.ylim([-7,0.0])
leg=plt.legend()

plt.draw() # Draw the figure so you can find the positon of the legend. 
# Get the bounding box of the original legend
bb = leg.get_bbox_to_anchor().transformed(ax.transAxes.inverted()) 
# Change to location of the legend. 
xOffset = 0.6
bb.x0 += xOffset
bb.x1 += xOffset
leg.set_bbox_to_anchor(bb, transform = ax.transAxes)
plt.show()

## 